In [ ]:
import importlib

import torch
from torch import Generator

import occhio
import occhio.toy_model as tm
from occhio.autoencoder import TiedLinearRelu
from occhio.distributions import SparseUniform
from occhio.model_grid import Axis, ModelGrid
from occhio.visualization.embedding import plot_embedding

In [ ]:
N_FEATURES = 10
N_HIDDEN = 2
FEATURE_IMPORTANCE_DECAY = 1

In [ ]:
def create_model(params):
    device = "mps"
    generator = Generator(device=device).manual_seed(42)

    return tm.ToyModel(
        distribution=SparseUniform(
            n_features=N_FEATURES,
            p_active=params["Feature Probability"],
            generator=generator,
            device=device,
        ),
        ae=TiedLinearRelu(
            n_features=N_FEATURES, n_hidden=N_HIDDEN, generator=generator, device=device
        ),
        importances=torch.tensor(
            [params["Relative Importance"] ** i for i in range(N_FEATURES)]
        ),
        device=device,
    )


model_grid = ModelGrid(
    create_model,
    axes=[
        Axis(label="Feature Probability", values=[0.001, 0.01, 0.1, 0.3, 1.0]),
        Axis(label="Relative Importance", values=[1, 0.7]),
    ],
)

In [ ]:
model_grid.fit()

In [ ]:
fig = plot_embedding(model_grid)

In [ ]:
fig.show()